In [2]:
import json
import torch
from torch.utils.data import Dataset

class EBMNLPDataset(Dataset):
    def __init__(self, json_file, tokenizer, max_length=512):
        with open(json_file, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
            
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        
        self.label_map = {
            None: 0,  
            "P": 1,   
            "I": 2,   
            "O": 3    
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['model_input_text']
        missing_slot = item['missing_slot']
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,    
            max_length=self.max_length, 
            padding='max_length',       
            truncation=True,            
            return_attention_mask=True, 
            return_tensors='pt'        
        )
        
        
        label = self.label_map[missing_slot]
        
        
        return {
            
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [3]:
import torch
from transformers import AutoTokenizer



print("Loading official BioBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")



train_dataset = EBMNLPDataset("ebm_nlp_train_mixed.json", tokenizer)
test_dataset = EBMNLPDataset("ebm_nlp_test_mixed.json", tokenizer)


def tokenize_and_save(dataset, output_filename):
    print(f"Processing and tokenizing data for {output_filename}...")
    tokenized_data = []
    
    for i in range(len(dataset)):
        tokenized_data.append(dataset[i])
     
    torch.save(tokenized_data, output_filename)
    print(f"Successfully saved {len(tokenized_data)} tokenized records to {output_filename}")


tokenize_and_save(train_dataset, "ebm_nlp_train_tokenized.pt")
tokenize_and_save(test_dataset, "ebm_nlp_test_tokenized.pt")

print("\n" + "="*40)
print("VERIFYING TENSOR SHAPES (Sample 0)")
print("="*40)
sample = train_dataset[0]

print(f"Input IDs shape:      {sample['input_ids'].shape}")
print(f"Attention Mask shape: {sample['attention_mask'].shape}")
print(f"Label tensor:         {sample['labels']} (Shape: {sample['labels'].shape})")

c:\Users\erank\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading official BioBERT tokenizer...


c:\Users\erank\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\erank\.cache\huggingface\hub\models--dmis-lab--biobert-base-cased-v1.2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Processing and tokenizing data for ebm_nlp_train_tokenized.pt...
Successfully saved 4792 tokenized records to ebm_nlp_train_tokenized.pt
Processing and tokenizing data for ebm_nlp_test_tokenized.pt...
Successfully saved 189 tokenized records to ebm_nlp_test_tokenized.pt

VERIFYING TENSOR SHAPES (Sample 0)
Input IDs shape:      torch.Size([512])
Attention Mask shape: torch.Size([512])
Label tensor:         0 (Shape: torch.Size([]))
